# Plant disease classification — Colab version

Same approach as `plant_disease_transfer.py` on the VM: **frozen pretrained backbone → cache feature vectors once → train a small linear head.**
The difference is that Colab gives us a GPU, so bigger backbones (ResNet34 and up) are cheap to try.

## One-time setup

1. Upload `archive.zip` (the dataset) to Google Drive at **`My Drive/ComputerVision/archive.zip`**.
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run the cells top to bottom.

To try a different backbone, change `BACKBONE_NAME` in the config cell and rerun from there.

In [ ]:
# Mount Google Drive. The dataset zip lives here, and so will the feature cache
# and saved heads — Colab's local disk (/content) is wiped when the runtime
# disconnects, Drive is not.
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Unzip to Colab's LOCAL disk, not Drive. Reading 1500 JPEGs directly from
# Drive is very slow (each file is a network round-trip); the local disk is not.
# This takes ~1 minute and must be redone after every runtime reset.
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/ComputerVision")
DATA_ROOT = Path("/content/data")

if not DATA_ROOT.exists():
    !unzip -q "{DRIVE_ROOT}/archive.zip" -d "{DATA_ROOT}"

for split in ["Train/Train", "Validation/Validation", "Test/Test"]:
    n = sum(len(files) for _, _, files in os.walk(DATA_ROOT / split))
    print(f"{split:<24} {n:>5} images")

Train/Train               1322 images
Validation/Validation       60 images
Test/Test                  150 images


In [ ]:
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models, transforms
from torchvision.models import (
    MobileNet_V3_Small_Weights,
    ResNet18_Weights,
    ResNet34_Weights,
)

torch.manual_seed(0)

# ---------------------------------------------------------------------------
# Config — change BACKBONE_NAME to try another model
# ---------------------------------------------------------------------------
BACKBONE_NAME = "resnet34"

SPLITS = {
    "train": DATA_ROOT / "Train" / "Train",
    "val": DATA_ROOT / "Validation" / "Validation",
    "test": DATA_ROOT / "Test" / "Test",
}
CACHE_DIR = DRIVE_ROOT / "feature_cache" / BACKBONE_NAME  # one folder per backbone
HEAD_PATH = DRIVE_ROOT / f"plant_disease_head_{BACKBONE_NAME}.pth"

IMAGE_SIZE = 224
EXTRACT_BATCH = 32
NUM_WORKERS = 2  # Colab usually gives 2 CPU cores for JPEG decoding

HEAD_EPOCHS = 40
HEAD_BATCH = 64
HEAD_LR = 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")
if device == "cpu":
    print("WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU")

Using cuda device


In [ ]:
# ---------------------------------------------------------------------------
# Backbones. Each entry: (constructor, name of the classifier attribute to
# remove). ResNets call it .fc, MobileNets call it .classifier. Replacing it
# with Identity makes forward() return the pooled feature vector.
#
# The feature width is read from the model rather than hard-coded, so adding
# a backbone here is a one-line change.
# ---------------------------------------------------------------------------
BACKBONES = {
    "mobilenet_v3_small": (
        lambda: models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1),
        "classifier",
    ),
    "resnet18": (lambda: models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1), "fc"),
    "resnet34": (lambda: models.resnet34(weights=ResNet34_Weights.IMAGENET1K_V1), "fc"),
}

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def build_backbone(name):
    constructor, head_attr = BACKBONES[name]
    backbone = constructor()
    old_head = getattr(backbone, head_attr)
    # The classifier's input width IS the feature width (works for both
    # a bare Linear and MobileNet's Sequential, whose first layer is Linear)
    first_linear = old_head if isinstance(old_head, nn.Linear) else old_head[0]
    feature_dim = first_linear.in_features
    setattr(backbone, head_attr, nn.Identity())
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    return backbone.to(device), feature_dim


@torch.no_grad()
def extract_features(backbone, split_name, split_path):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CACHE_DIR / f"{split_name}.pt"

    if cache_file.exists():
        blob = torch.load(cache_file, weights_only=True)
        if blob.get("backbone") != BACKBONE_NAME:
            raise RuntimeError(
                f"{cache_file} holds {blob.get('backbone')} features, "
                f"not {BACKBONE_NAME}. Delete it and rerun."
            )
        print(f"  {split_name:<6} loaded {len(blob['labels']):>5} cached features")
        return blob["features"], blob["labels"], blob["classes"]

    dataset = datasets.ImageFolder(split_path, transform=preprocess)
    loader = DataLoader(dataset, batch_size=EXTRACT_BATCH, shuffle=False, num_workers=NUM_WORKERS)

    feats, labels = [], []
    start = time.perf_counter()
    for images, y in loader:
        feats.append(backbone(images.to(device)).cpu())
        labels.append(y)
    elapsed = time.perf_counter() - start

    feats, labels = torch.cat(feats), torch.cat(labels)
    torch.save(
        {"features": feats, "labels": labels, "classes": dataset.classes, "backbone": BACKBONE_NAME},
        cache_file,
    )
    print(f"  {split_name:<6} extracted {len(labels):>5} images in {elapsed:>6.1f}s "
          f"({len(labels) / elapsed:.1f} img/s)")
    return feats, labels, dataset.classes


def evaluate(head, features, labels):
    head.eval()
    with torch.no_grad():
        preds = head(features.to(device)).argmax(1).cpu()
    return (preds == labels).float().mean().item(), preds

In [ ]:
# ---------------------------------------------------------------------------
# 1. Feature extraction — the expensive step, paid once per backbone
# ---------------------------------------------------------------------------
print(f"Extracting features with frozen {BACKBONE_NAME}")
print("-" * 60)
backbone, FEATURE_DIM = build_backbone(BACKBONE_NAME)

cached = {}
for name, path in SPLITS.items():
    feats, labels, classes = extract_features(backbone, name, path)
    cached[name] = (feats, labels)

print(f"\nClasses: {classes}")
print(f"Feature dim: {FEATURE_DIM}   train shape: {tuple(cached['train'][0].shape)}")

Extracting features with frozen resnet34
------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 124MB/s]


  train  extracted  1322 images in  170.4s (7.8 img/s)
  val    extracted    60 images in    8.4s (7.1 img/s)
  test   extracted   150 images in   20.1s (7.5 img/s)

Classes: ['Healthy', 'Powdery', 'Rust']
Feature dim: 512   train shape: (1322, 512)


In [ ]:
# ---------------------------------------------------------------------------
# 2. Train the linear head on cached features — the cheap step
# ---------------------------------------------------------------------------
train_x, train_y = cached["train"]
val_x, val_y = cached["val"]
test_x, test_y = cached["test"]

head = nn.Linear(FEATURE_DIM, len(classes)).to(device)
optimizer = torch.optim.Adam(head.parameters(), lr=HEAD_LR)
loss_fn = nn.CrossEntropyLoss()
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=HEAD_BATCH, shuffle=True)

print(f"Training head ({FEATURE_DIM} -> {len(classes)})")
print("-" * 60)
for epoch in range(1, HEAD_EPOCHS + 1):
    head.train()
    epoch_loss = 0.0
    for bx, by in train_loader:
        loss = loss_fn(head(bx.to(device)), by.to(device))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
    if epoch % 5 == 0 or epoch == 1:
        val_acc, _ = evaluate(head, val_x, val_y)
        print(f"  epoch {epoch:>3}  loss {epoch_loss / len(train_loader):.4f}  val acc {val_acc * 100:.1f}%")

Training head (512 -> 3)
------------------------------------------------------------
  epoch   1  loss 0.8321  val acc 90.0%
  epoch   5  loss 0.1766  val acc 95.0%
  epoch  10  loss 0.1121  val acc 96.7%
  epoch  15  loss 0.0866  val acc 95.0%
  epoch  20  loss 0.0722  val acc 96.7%
  epoch  25  loss 0.0604  val acc 95.0%
  epoch  30  loss 0.0537  val acc 96.7%
  epoch  35  loss 0.0454  val acc 95.0%
  epoch  40  loss 0.0402  val acc 96.7%


In [ ]:
# ---------------------------------------------------------------------------
# 3. Final evaluation
# ---------------------------------------------------------------------------
test_acc, test_preds = evaluate(head, test_x, test_y)
val_acc, _ = evaluate(head, val_x, val_y)

print("=" * 60)
print(f"RESULTS — {BACKBONE_NAME}")
print("=" * 60)
print(f"Validation accuracy: {val_acc * 100:.1f}%")
print(f"Test accuracy:       {test_acc * 100:.1f}%   "
      f"({int((test_preds != test_y).sum())} errors of {len(test_y)})")

print("\nPer-class accuracy (test):")
for i, c in enumerate(classes):
    mask = test_y == i
    print(f"  {c:<10} {(test_preds[mask] == test_y[mask]).float().mean() * 100:>5.1f}%")

print("\nConfusion matrix (rows = actual, cols = predicted):")
confusion = torch.zeros(len(classes), len(classes), dtype=torch.int32)
for a, p in zip(test_y, test_preds):
    confusion[a, p] += 1
print(" " * 12 + "".join(f"{c[:8]:>10}" for c in classes))
for i, c in enumerate(classes):
    print(f"  {c:<10}" + "".join(f"{confusion[i, j].item():>10}" for j in range(len(classes))))

torch.save({"head": head.state_dict(), "classes": classes, "backbone": BACKBONE_NAME}, HEAD_PATH)
print(f"\nSaved head to {HEAD_PATH}")

RESULTS — resnet34
Validation accuracy: 96.7%
Test accuracy:       96.0%   (6 errors of 150)

Per-class accuracy (test):
  Healthy     98.0%
  Powdery     92.0%
  Rust        98.0%

Confusion matrix (rows = actual, cols = predicted):
               Healthy   Powdery      Rust
  Healthy           49         0         1
  Powdery            3        46         1
  Rust               0         1        49

Saved head to /content/drive/MyDrive/ComputerVision/plant_disease_head_resnet34.pth


## Reading the result

- 150 test images → **one image = 0.67 percentage points**. A gap smaller than ~3 images is noise.
- On the VM, MobileNetV3-Small and ResNet18 both scored exactly **96.0% (6 errors)**, stable across 5 seeds.
- ResNet34 has the **same 512-wide output** as ResNet18 — it's the same building blocks, just deeper. If it doesn't beat 96.0% by a clear margin, the bottleneck is the data (mild Powdery cases), not the network.